In [1]:
# ----------------------------
# Imports & configuration
# ----------------------------
import sys
import glob
import re
import h5py
import dataclasses
import sys
import glob

import os
from tqdm import tqdm
import importlib

from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
# ----------------------------
# Imports & configuration
# ----------------------------

INP = Path("/storage/home/hcoda1/4/mugliotti3/scratch/temporary/DNS_SPIDER.nc")   #where DNS is saved
ds = xr.open_dataset(INP)                #import DNS
w0 = np.asarray(ds["w"].values)
Nt, Nx, Ny = w0.shape
print(Nx, Ny, Nt)

dx = 2*np.pi/Nx 
dy = 2*np.pi/Ny
dt = float(ds.attrs["save_every"])
print(dx, dy, dt)

Delta = 6*np.pi/256

2048 2048 50
0.0030679615757712823 0.0030679615757712823 0.01


In [3]:
# ----------------------------
# Some functions we need
# ----------------------------

def fourier_filter_np(field: np.ndarray, Delta: float) -> np.ndarray:
    field = np.asarray(field)
    Nx, Ny = field.shape[:2]

    # integer wavenumber indices consistent with np.fft.fft2 ordering
    qx = np.fft.fftfreq(Nx) * Nx  # [0,1,...,Nx/2,-Nx/2+1,...,-1]
    qy = np.fft.fftfreq(Ny) * Ny

    KX, KY = np.meshgrid(qx, qy, indexing="ij")
    R2 = KX**2 + KY**2

    G = np.exp(-(R2) * (Delta**2) / 24.0)

    # Apply filter with vectorized FFTs over trailing dims
    if field.ndim == 2:
        F = np.fft.fft2(field, s=(Nx, Ny))
        out = np.real(np.fft.ifft2(F * G, s=(Nx, Ny)))
        return out

    elif field.ndim == 3:
        # (Nx,Ny,K)
        F = np.fft.fft2(field, s=(Nx, Ny), axes=(0, 1))
        out = np.real(np.fft.ifft2(F * G[..., None], s=(Nx, Ny), axes=(0, 1)))
        return out

    elif field.ndim == 4:
        # (Nx,Ny,K,C)
        F = np.fft.fft2(field, s=(Nx, Ny), axes=(0, 1))
        out = np.real(np.fft.ifft2(F * G[..., None, None], s=(Nx, Ny), axes=(0, 1)))
        return out

    else:
        raise ValueError("ERROR: field must have 2D, 3D, or 4D shape")

def vorticity_to_velocity_np(w0: np.ndarray, Lx: float = 2*np.pi, Ly: float = 2*np.pi):
    w0 = np.asarray(w0, dtype=np.float64)
    if w0.ndim != 3:
        raise ValueError(f"Expected w0 with shape (Nt,Nx,Ny), got {w0.shape}")
    Nt, Nx, Ny = w0.shape
    dx = Lx / Nx
    dy = Ly / Ny
    kx = 2.0 * np.pi * np.fft.fftfreq(Nx, d=dx)   # (Nx,)
    ky = 2.0 * np.pi * np.fft.fftfreq(Ny, d=dy)   # (Ny,)
    KX = kx[:, None]
    KY = ky[None, :] 
    K2 = KX**2 + KY**2
    w_hat = np.fft.fft2(w0, axes=(1, 2))  # (Nt,Nx,Ny)
    K2_safe = K2.copy()
    K2_safe[0, 0] = 1.0
    psi_hat = -w_hat / K2_safe[None, :, :]
    psi_hat[:, 0, 0] = 0.0  # set mean mode to 0
    u_hat = (1j * KY[None, :, :]) * psi_hat
    v_hat = (-1j * KX[None, :, :]) * psi_hat
    U = np.real(np.fft.ifft2(u_hat, axes=(1, 2)))  # (Nt,Nx,Ny)
    V = np.real(np.fft.ifft2(v_hat, axes=(1, 2)))

    return U, V  

def pressure_poisson_from_velocity_np(u_bar: np.ndarray, Lx: float = 2*np.pi, Ly: float = 2*np.pi):
    u_bar = np.asarray(u_bar, dtype=np.float64)
    if u_bar.ndim != 4 or u_bar.shape[-1] != 2:
        raise ValueError(f"Expected u_bar with shape (Nx,Ny,Nt,2), got {u_bar.shape}")
    Nx, Ny, Nt, _ = u_bar.shape
    dx = Lx / Nx
    dy = Ly / Ny
    kx = 2.0 * np.pi * np.fft.fftfreq(Nx, d=dx)  # (Nx,)
    ky = 2.0 * np.pi * np.fft.fftfreq(Ny, d=dy)  # (Ny,)
    KX = kx[:, None]                              # (Nx,1)
    KY = ky[None, :]                              # (1,Ny)
    K2 = KX**2 + KY**2
    u = u_bar[..., 0]  # (Nx,Ny,Nt)
    v = u_bar[..., 1]  # (Nx,Ny,Nt)
    uu = u * u
    uv = u * v
    vv = v * v
    UU_hat = np.fft.fft2(uu, axes=(0, 1))  # (Nx,Ny,Nt)
    UV_hat = np.fft.fft2(uv, axes=(0, 1))
    VV_hat = np.fft.fft2(vv, axes=(0, 1))
    rhs_hat = (KX**2)[:, :, None] * UU_hat + 2.0 * (KX * KY)[:, :, None] * UV_hat + (KY**2)[:, :, None] * VV_hat
    K2_safe = K2.copy()
    K2_safe[0, 0] = 1.0
    p_hat = -rhs_hat / K2_safe[:, :, None]
    p_hat[0, 0, :] = 0.0  # zero-mean pressure gauge
    p_bar = np.real(np.fft.ifft2(p_hat, axes=(0, 1)))  # (Nx,Ny,Nt)
    return p_bar

In [4]:
# ----------------------------
# Calculate u and p filtered
# ----------------------------

Nt, Nx, Ny = w0.shape  
U_txy, V_txy = vorticity_to_velocity_np(w0, Lx=2*np.pi, Ly=2*np.pi)  

U = np.transpose(U_txy, (1, 2, 0))  
V = np.transpose(V_txy, (1, 2, 0))  

u = np.stack((U, V), axis=-1)       
u_bar = fourier_filter_np(u, Delta=Delta)  
p_bar = pressure_poisson_from_velocity_np(u_bar, Lx=2*np.pi, Ly=2*np.pi)  

In [5]:
# ----------------------------
# Define the SGS stress tensor tau
# ----------------------------

T_xx = fourier_filter_np(u[:,:,:,0]**2, Delta)-u_bar[:,:,:,0]**2 
T_xy = fourier_filter_np(u[:,:,:,0]*u[:,:,:,1], Delta)-u_bar[:,:,:,0]*u_bar[:,:,:,1]
T_yy = fourier_filter_np(u[:,:,:,1]**2, Delta)-u_bar[:,:,:,1]**2 
T = np.stack([np.stack([T_xx, T_xy], axis=-1),  np.stack([T_xy, T_yy], axis=-1)], axis=-2)
del T_xx, T_xy, T_yy

In [6]:
# ----------------------------
# Define the library
# ----------------------------

#Note: Picking correct hyperparameters is very hard and something that requires lots of training and developing intuition.
#I would argue, it's easier to just try a few different values at first rather than trying to find the perfect value beforehand.
#Also always feel free to reach out to me if it's not working out: matteougliotti@gatech.edu

from utils import save, load
from library import *
from continuous_process_library_terms import *
from commons_utils import *
from commons_identify_models import *

Uobs = Observable(string='u', rank=1)
Pobs = Observable(string='p', rank=0)
Tobs = Observable(string='T', rank=2, can_commute_indices = True, antisymmetric = False)
observables = [Uobs,Tobs,Pobs]
data_dict = {'u' : u_bar ,'T': T, 'p':p_bar}

np.random.seed(1)
world_size = np.array(u_bar.shape[:3])
pad = 0
# fix random seed
dxs = [dx, dy, dt]
max_observable_counts = {Uobs: 2, Tobs:1, Pobs:1}
srd = SRDataset(world_size=world_size, data_dict=data_dict, observables=observables, dxs=dxs, 
                irreps=SRDataset.all_rank2_irreps(), cache_primes=True)
srd.make_libraries(max_complexity=4, max_observable_counts= max_observable_counts, max_dt = 1)

In [7]:
# ----------------------------
# Calculate the library values
# ----------------------------

import os

num_cores = os.cpu_count()
print(f"Logical CPU cores available: {num_cores}")
Ndomainz = len(srd.libs[SymmetricTraceFree(rank=2)].terms)*10
num_processorz = num_cores - 4
Ndomainz_rounded = num_processorz * round(Ndomainz / num_processorz)
print(len(srd.libs[SymmetricTraceFree(rank=2)].terms))
print(num_processorz)
print(Ndomainz_rounded)
dom_width = 16
dom_time = 20 
pad = 0
srd.make_domains(ndomains=Ndomainz_rounded, domain_size=[dom_width, dom_width, dom_time], pad=pad)
srd.make_weights(m=12, qmax=0)
srd.set_LT_scale(L=dx*dom_width, T=dt*dom_time) # note that this line must go before make_library_matrices
srd.make_library_matrices(debug=False, parallel=True, num_processors=num_processorz)  # or whatever number of cores you want

Logical CPU cores available: 128
45
124
496


In [8]:
# ----------------------------
# Look at the terms you are interested in and not down their index (here e.g. 0 for T_ab)
# ----------------------------

srd.libs[srd.irreps[3]].terms

[T_αβ,
 T_αβ · p,
 T_αβ · ∂t p,
 T_γγ · u_α · u_β,
 T_αβ · u_γ · u_γ,
 T_αγ · u_β · u_γ,
 T_γγ · ∂α u_β,
 T_αβ · ∂γ u_γ,
 T_αγ · ∂β u_γ,
 T_αγ · ∂γ u_β,
 ∂γ T_αβ · u_γ,
 ∂γ T_αγ · u_β,
 ∂α T_βγ · u_γ,
 ∂α T_γγ · u_β,
 ∂γ² T_αβ,
 ∂α ∂β T_γγ,
 ∂α ∂γ T_βγ,
 ∂t T_αβ,
 ∂t T_αβ · p,
 p · u_α · u_β,
 p · u_α · ∂t u_β,
 p · ∂α u_β,
 p · ∂t ∂α u_β,
 ∂α p · u_β,
 ∂α p · ∂t u_β,
 ∂α ∂β p,
 ∂t p · u_α · u_β,
 ∂t p · ∂α u_β,
 ∂t ∂α p · u_β,
 ∂t ∂α ∂β p,
 u_α · u_β,
 u_γ · ∂α ∂γ u_β,
 u_γ · ∂α ∂β u_γ,
 u_α · ∂β ∂γ u_γ,
 u_α · ∂γ² u_β,
 u_α · ∂t u_β,
 ∂α u_β,
 ∂α u_β · ∂γ u_γ,
 ∂α u_γ · ∂β u_γ,
 ∂γ u_α · ∂γ u_β,
 ∂α u_γ · ∂γ u_β,
 ∂α ∂γ² u_β,
 ∂α ∂β ∂γ u_γ,
 ∂t u_α · ∂t u_β,
 ∂t ∂α u_β]

In [9]:
# ----------------------------
# Run the Regression
# ----------------------------

from commons_identify_models import *
import copy

libs = srd.libs
lib3 = libs[srd.irreps[3]]

for aaaa in range(1,4):
    max_kk = 6
    print(aaaa)
    reg_opts_list = []

    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold
    scaler = Scaler(sub_inds=None, char_sizes=lib3.col_weights, row_norms=None, train_fraction=1, unit_rows=True)
    #init = Initializer(method='combinatorial', start_k=2)
    #init = Initializer(method='combinatorial', start_k=9999)
    init = Initializer(method='combinatorial', start_k=max_kk)
    #res = Residual(residual_type='fixed_column', anchor_col=0)
    #res = Residual(residual_type='dominant_balance')
    res = Residual(residual_type='dominant_balance')

    iterator = ModelIterator(max_k=max_kk, backward_forward=True, max_passes=20)
    thres = Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=aaaa)
    #thres = Threshold(threshold_type='information', ic=AIC)

    opts = {'scaler': scaler, 'initializer': init, 'residual': res,
            'model_iterator': iterator, 'threshold': thres}
    opts['verbose'] = False
    opts['inhomog'] = True
    opts['inhomog_col'] = 0 #25

    reg_result = sparse_reg_bf(lib3.Q, **opts)
    zipped = [(lib3.terms[i], c) for i, c in enumerate(reg_result.xi) if c != 0]
    eqn = Equation([e[0] for e in zipped], [e[1] for e in zipped])

    print(eqn, "; residual:", reg_result.lambd)

1
T_αβ = 0 ; residual: 0.997259598966036
2
T_αβ + -0.0004528635441748187 · ∂γ u_α · ∂γ u_β = 0 ; residual: 0.002904397664351813
3
T_αβ + 5.130861284610395e-05 · ∂α ∂β T_γγ + -0.00045440156091298994 · ∂γ u_α · ∂γ u_β = 0 ; residual: 0.0025627240923804034


In [10]:
# ----------------------------
# Compare coefficient found to theoretical NGS coefficient
# ----------------------------

Delta**2/12

0.0004517946350596325

In [11]:
# ----------------------------
# Calculate NGM2 and subtract it from Tau
# ----------------------------

Lx = 2*np.pi
Ly = 2*np.pi
dx = Lx / Nx
dy = Ly / Ny

kx = 2*np.pi * np.fft.fftfreq(Nx, d=dx)
ky = 2*np.pi * np.fft.fftfreq(Ny, d=dy)
KX = kx[:, None]
KY = ky[None, :]

def spectral_grad_np(field):
    F = np.fft.fft2(field, axes=(0,1))
    dfdx = np.real(np.fft.ifft2(1j * KX[..., None] * F, axes=(0,1)))
    dfdy = np.real(np.fft.ifft2(1j * KY[..., None] * F, axes=(0,1)))
    return dfdx, dfdy

dux_dx, dux_dy = spectral_grad_np(u_bar[..., 0])
duy_dx, duy_dy = spectral_grad_np(u_bar[..., 1])

coef = Delta**2 / 12.0
NGM2_xx = coef * (dux_dx**2 + dux_dy**2)
NGM2_xy = coef * (dux_dx*duy_dx + dux_dy*duy_dy)
NGM2_yy = coef * (duy_dx**2 + duy_dy**2)
NGM2 = np.stack([np.stack([NGM2_xx, NGM2_xy], axis=-1),np.stack([NGM2_xy, NGM2_yy], axis=-1),],axis=-2,)
del NGM2_xx, NGM2_xy, NGM2_yy

T2 = T - NGM2
del NGM2

In [12]:
# ----------------------------
# Calculate the library values
# ----------------------------

from utils import save, load
from library import *
from continuous_process_library_terms import *
from commons_utils import *
from commons_identify_models import *

Uobs = Observable(string='u', rank=1)
Pobs = Observable(string='p', rank=0)
Tobs = Observable(string='T', rank=2, can_commute_indices = True, antisymmetric = False)
observables = [Uobs,Tobs,Pobs]
data_dict = {'u' : u_bar ,'T': T2, 'p':p_bar}

np.random.seed(1)
world_size = np.array(u_bar.shape[:3])
pad = 0
# fix random seed
dxs = [dx, dy, dt]
max_observable_counts = {Uobs: 2, Tobs:1, Pobs:1}
srd = SRDataset(world_size=world_size, data_dict=data_dict, observables=observables, dxs=dxs, 
                irreps=SRDataset.all_rank2_irreps(), cache_primes=True)
srd.make_libraries(max_complexity=6, max_observable_counts= max_observable_counts, max_dt = 0)

In [13]:
# ----------------------------
# Calculate the library values
# ----------------------------

import os

num_cores = os.cpu_count()
print(f"Logical CPU cores available: {num_cores}")
Ndomainz = len(srd.libs[SymmetricTraceFree(rank=2)].terms)*10
num_processorz = num_cores - 4
Ndomainz_rounded = num_processorz * round(Ndomainz / num_processorz)
print(len(srd.libs[SymmetricTraceFree(rank=2)].terms))
print(num_processorz)
print(Ndomainz_rounded)
dom_width = 16
dom_time = 20 
pad = 0
srd.make_domains(ndomains=Ndomainz_rounded, domain_size=[dom_width, dom_width, dom_time], pad=pad)
srd.make_weights(m=12, qmax=0)
srd.set_LT_scale(L=dx*dom_width, T=dt*dom_time) # note that this line must go before make_library_matrices
srd.make_library_matrices(debug=False, parallel=True, num_processors=num_processorz)  # or whatever number of cores you want

Logical CPU cores available: 128
244
124
2480


In [14]:
# ----------------------------
# Look at the terms you are interested in and not down their index (here e.g. 0 for T_ab)
# ----------------------------

srd.libs[srd.irreps[3]].terms

[T_αβ,
 T_αβ · p,
 T_αγ · p · u_β · u_γ,
 T_γγ · p · u_α · u_β,
 T_αβ · p · u_γ · u_γ,
 T_αγ · p · ∂β u_γ,
 T_γγ · p · ∂α u_β,
 T_αγ · p · ∂γ u_β,
 T_αβ · p · ∂γ u_γ,
 T_αβ · ∂γ p · u_γ,
 T_γγ · ∂α p · u_β,
 T_αγ · ∂γ p · u_β,
 T_αγ · ∂β p · u_γ,
 T_αβ · ∂γ² p,
 T_αγ · ∂β ∂γ p,
 T_γγ · ∂α ∂β p,
 T_γγ · u_α · u_β,
 T_αβ · u_γ · u_γ,
 T_αγ · u_β · u_γ,
 T_αγ · u_β · ∂δ² u_γ,
 T_αγ · u_β · ∂γ ∂δ u_δ,
 T_αγ · u_γ · ∂δ² u_β,
 T_αγ · u_δ · ∂γ ∂δ u_β,
 T_αγ · u_δ · ∂β ∂δ u_γ,
 T_αγ · u_γ · ∂β ∂δ u_δ,
 T_αβ · u_γ · ∂δ² u_γ,
 T_αβ · u_γ · ∂γ ∂δ u_δ,
 T_αγ · u_δ · ∂β ∂γ u_δ,
 T_γδ · u_γ · ∂α ∂β u_δ,
 T_γγ · u_δ · ∂α ∂β u_δ,
 T_γδ · u_γ · ∂α ∂δ u_β,
 T_γγ · u_δ · ∂α ∂δ u_β,
 T_γγ · u_α · ∂δ² u_β,
 T_γδ · u_α · ∂γ ∂δ u_β,
 T_γδ · u_α · ∂β ∂γ u_δ,
 T_γγ · u_α · ∂β ∂δ u_δ,
 T_αγ · ∂β u_γ,
 T_γγ · ∂α u_β,
 T_αβ · ∂γ u_γ,
 T_αγ · ∂γ u_β,
 T_αγ · ∂γ u_β · ∂δ u_δ,
 T_γγ · ∂δ u_α · ∂δ u_β,
 T_γδ · ∂γ u_α · ∂δ u_β,
 T_αγ · ∂γ u_δ · ∂δ u_β,
 T_γγ · ∂α u_δ · ∂δ u_β,
 T_γδ · ∂α u_γ · ∂δ u_β,
 T_αγ · ∂β u_δ ·

In [15]:
# ----------------------------
# Run the regression
# ----------------------------

from commons_identify_models import *
import copy

libs = srd.libs
lib3 = libs[srd.irreps[3]]

for aaaa in range(1,4):
    max_kk = 6
    print(aaaa)
    reg_opts_list = []

    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold
    scaler = Scaler(sub_inds=None, char_sizes=lib3.col_weights, row_norms=None, train_fraction=1, unit_rows=True)
    #init = Initializer(method='combinatorial', start_k=2)
    #init = Initializer(method='combinatorial', start_k=9999)
    init = Initializer(method='power', start_k=max_kk)
    #res = Residual(residual_type='fixed_column', anchor_col=0)
    #res = Residual(residual_type='dominant_balance')
    res = Residual(residual_type='dominant_balance')

    iterator = ModelIterator(max_k=max_kk, backward_forward=True, max_passes=20)
    thres = Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=aaaa)
    #thres = Threshold(threshold_type='information', ic=AIC)

    opts = {'scaler': scaler, 'initializer': init, 'residual': res,
            'model_iterator': iterator, 'threshold': thres}
    opts['verbose'] = False
    opts['inhomog'] = True
    opts['inhomog_col'] = 0 #25

    reg_result = sparse_reg_bf(lib3.Q, **opts)
    zipped = [(lib3.terms[i], c) for i, c in enumerate(reg_result.xi) if c != 0]
    eqn = Equation([e[0] for e in zipped], [e[1] for e in zipped])

    print(eqn, "; residual:", reg_result.lambd)

1
T_αβ = 0 ; residual: 0.03152612479725029
2
T_αβ + -1.0314982415201128e-07 · ∂γ ∂δ u_α · ∂γ ∂δ u_β = 0 ; residual: 5.234117420985081e-05
3
T_αβ + 3.8380353509313194e-05 · ∂α ∂β T_γγ + -1.0369776274491536e-07 · ∂γ ∂δ u_α · ∂γ ∂δ u_β = 0 ; residual: 4.536738065588321e-05


In [16]:
# ----------------------------
# Compare coefficient found to theoretical NGS coefficient
# ----------------------------

Delta**4/288

1.0205919613433324e-07